# CNPJ × party extraction and PF anonymization — empirical validation

Prototype for:
1. **CNPJ extraction** from court decision text (`processos_delta.decisao`) linked to FACE parties.
2. **PF anonymization** as a derived view (`decisao_anonimizada`) without mutating raw text.

Implementation modules:
- `src/utils/cnpj_extraction.py` — syntax-first CNPJ rules
- `src/utils/pf_redaction.py` — deterministic PF redaction
- `src/utils/legalbert_ner.py` — optional LegalBERT recall layer

**Outputs:**
- `notebooks/playground/output/cnpj_validation_sample.csv`
- `notebooks/playground/output/pf_redaction_validation_sample.csv`


## Extraction rules (data paper reference)

TJSP fiscal execution decisions repeat a **court header** before the merits. CNPJ placement
in that header determines the owning party. Do **not** score the whole snippet with fuzzy
name matching — the snippet almost always names **both** creditor and debtor, which biases
toward `autores` (Prefeitura / FESP / Município).

### Pre-extraction filters (applied before any pattern matching)

Two mandatory rejection gates run **before** any pattern is attempted.
If either fires, the candidate number is dropped entirely (`cnpj = null`).

#### 1. CDA false-positive filter (`is_cda_number`)

São Paulo IPTU executions include a **CDA inscription number** (Número de Inscrição
da Certidão de Dívida Ativa) in the court header. This 14-digit enrollment ID is
structurally identical to a CNPJ and can pass check-digit validation, producing
systematic false positives.

Signature in text:
```text
Executado: Yasutaka Fukui
CDA/Outros nº: 51214262022401, 51214262022402, …
```

The regex `\d{2}[.]\d{3}[.]\d{3}/\d{4}-\d{2}` matches the first CDA number as
`51.214.262/0224-01`. Rule: if the candidate's raw digits appear verbatim inside a
`CDA/Outros nº:` list in the same snippet → **reject**.

#### 2. Natural-person filter (`is_natural_person`)

CNPJs identify **legal entities** (Pessoa Jurídica, PJ). If the party name resolved
from either the syntactic match or FACE contains **no legal-entity indicator**, the
attributed party is a natural person (Pessoa Física, PF) whose identifier is a CPF
(11 digits, never matched by the 14-digit CNPJ regex). Extracting a CNPJ for a PF
party is a logic error → **set `cnpj = null`**.

PJ indicators checked: `Ltda`, `S.A.`, `S/A`, `ME`, `EPP`, `EIRELI`, `CIA.`,
`Companhia`, `Empresa`, `Banco`, `Construtora`, `Incorporadora`, `Imobiliária`,
`Instituto`, `Fundação`, `Associação`, `Cooperativa`, `Logística`, `Holding`,
`Participações`, `Prefeitura`, `Município`, `Fazenda`, and related terms.

---

### Decision tree (apply in order, after filters pass)

| Priority | Pattern ID | Trigger (regex summary) | CNPJ owner | FACE pole |
|----------|------------|-------------------------|------------|----------|
| 1 | `payment_creditor` | `creditado` / `levantamento` … `CNPJ` | Creditor bank account | `autores` |
| 2 | `substitution_creditor` | `União Federal` / `Fazenda Nacional` … `CNPJ` | New creditor (substitution) | `autores` |
| 3 | `creditor_header_cnpj` | `Exequente: {name}, CNPJ` … **before** `Executado:` | Creditor on same line | `autores` |
| 4 | `debtor_header_block` | `Exequente:` … `Executado: {name} CNPJ:` | **Debtor** `{name}` | `reus` |
| 5 | `debtor_requerido` | `Requerente:` … `Requerido: {name} CNPJ:` | **Debtor** `{name}` | `reus` |
| 6 | `debtor_executado_inline` | `Executado: {name} CNPJ:` (no paired header in snippet) | **Debtor** | `reus` |
| 7 | `debtor_executado_cnpj_first` | `Executado: CNPJ: {cnpj} {name}` (inverted order) | **Debtor** | `reus` |
| 8 | `debtor_contra` | `contra CNPJ:` (debtor without explicit name) | Debtor (unnamed) | `reus` |
| 9 | `debtor_address_block` | `CEP … CNPJ:` or `CNPJ … com endereço` | Debtor (cadastral record) | `reus` |
| 10 | `debtor_inline_dash` | `{name} – CNPJ nº {cnpj}` in decision body | **Debtor** inline reference | `reus` |
| 11 | `debtor_name_comma_cnpj` | `{full name}, CNPJ {cnpj}` in decision body | **Debtor** after full name | `reus` |
| 12 | `jaccard_fallback` | None of the above | Best fuzzy match (low confidence) | either |

### Syntactic templates (examples from sample)

**Debtor header (~60% of mentions)** — CNPJ immediately follows the debtor label:

```text
Exequente: Município de Santana de Parnaíba
Executado: Convergas Equipamentos e Serviços Ltda
CNPJ: 68.347.343/0005-94
```

Legacy label variant (`Requerente` / `Requerido`):

```text
Requerente: Fazenda do Estado de São Paulo
Requerido: Danfat Industria e Comercio Ltda
CNPJ: 01.531.421/0001-01
```

**Inverted debtor header** — CNPJ precedes the name (Santana de Parnaíba variant):

```text
Executado: CNPJ: 07439331000171 Aol System Consultoria e Assessoria Em Informatica Ltda
```

**Debtor address block** — cadastral registration block:

```text
CNPJ 09047092000130, com endereço à Rua … CEP … São Paulo - SP
```

**Inline dash reference** — company cited in decision body:

```text
junto à proprietária Arquiville Desenvolvimento Imobiliário Ltda – CNPJ nº 04.178.194/0001-70
```

**Full name + comma** — company repeated in full then CNPJ:

```text
CDHU, COMPANHIA DE DESENVOLVIMENTO HABITACIONAL E URBANO DO ESTADO DE SÃO PAULO - CDHU, CNPJ 47.865.597/0001-09
```

**Creditor payment CNPJ** — municipality banking details after satisfaction:

```text
… creditado em conta … do Município de Arandu, CNPJ 46.634.176/0001-04, Banco do Brasil …
```

**Creditor exception** — CNPJ attached to exequente line before debtor appears:

```text
Exequente: SAEC, CNPJ 10.559.279/0001-00
Executado: RESIDENCIAL ESPLANADA
```

### Confidence levels

| Level | Condition |
|-------|-----------|
| `high` | Syntax pattern matched **and** `party_from_syntax` aligns with FACE (`token_jaccard ≥ 0.20` or substring) |
| `medium` | Syntax pattern matched but weak FACE alignment |
| `low` | `jaccard_fallback` only |
| `null` | CDA false positive **or** natural person (PF) — CNPJ field set to `null` |

### Proposed silver column (future)

| Column | Description |
|--------|-------------|
| `cnpj` | Normalized 14-digit ID with check digits validated; `null` for PF debtors or CDA numbers |
| `cnpj_syntax_pattern` | Pattern ID from decision tree above |
| `cnpj_party_name` | Name span parsed from decision text |
| `cnpj_face_pole` | `autores` or `reus` |
| `cnpj_confidence` | `high` / `medium` / `low` |
| `cnpj_rejected_reason` | `cda_number` / `natural_person` / `null` if not rejected |

In [1]:
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import polars as pl
from IPython.display import display

from config.paths import REPO_ROOT, SILVER_FACE_CLEAN, SILVER_PROCESSOS

OUTPUT_DIR = REPO_ROOT / "projects/litigancia/notebooks/playground/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 500
CONTEXT_CHARS = 120
RANDOM_SEED = 42
FACE_MATCH_THRESHOLD = 0.20

lf_processos = pl.scan_delta(str(SILVER_PROCESSOS))
lf_face = pl.scan_delta(str(SILVER_FACE_CLEAN))


## 1. CNPJ validation and syntax-first attribution

In [2]:
from utils.cnpj_extraction import (
    CnpjMention,
    DEFAULT_CONTEXT_CHARS as CONTEXT_CHARS,
    DEFAULT_FACE_MATCH_THRESHOLD as FACE_MATCH_THRESHOLD,
    extract_mentions,
)
from utils.pf_redaction import anonymize_decision, build_redaction_validation_rows
from utils.legalbert_ner import legalbert_available


## 2. Sample decisions with CNPJ-like patterns

In [3]:
CNPJ_LIKE = r"\d{2}[.]?\d{3}[.]?\d{3}/?\d{4}-?\d{2}"

df_pool = (
    lf_processos.filter(pl.col("decisao").str.contains(CNPJ_LIKE))
    .select("cd_processo", "id_processo", "decisao", "assunto", "comarca")
    .join(
        lf_face.select("cd_processo", "autores", "reus"),
        on="cd_processo",
        how="left",
    )
    .head(SAMPLE_SIZE * 3)
    .collect()
)

df_sample = df_pool.sample(
    n=min(SAMPLE_SIZE, len(df_pool)),
    shuffle=True,
    seed=RANDOM_SEED,
)
print(f"Sampled decision rows: {len(df_sample):,}")
print(f"With FACE parties: {df_sample.filter(pl.col('autores').is_not_null()).height:,}")

Sampled decision rows: 500
With FACE parties: 500


## 3. Extract mentions and summarize by syntax pattern

In [4]:
rows: list[dict] = []
for row in df_sample.iter_rows(named=True):
    for mention in extract_mentions(
        cd_processo=row["cd_processo"],
        id_processo=row["id_processo"],
        decisao=row["decisao"],
        autores_raw=row.get("autores"),
        reus_raw=row.get("reus"),
    ):
        rows.append(
            {
                **mention.__dict__,
                "assunto": row["assunto"],
                "comarca": row["comarca"],
                "pole_matches_syntax": (
                    mention.face_pole == mention.expected_face_pole
                    if mention.expected_face_pole and mention.face_pole
                    else None
                ),
                "manual_label": "",
                "notes": "",
            }
        )

df_mentions = pl.DataFrame(rows)
n_valid = df_mentions.filter(pl.col("cnpj").is_not_null()).height
n_rejected = df_mentions.filter(pl.col("cnpj").is_null()).height
print(f"Total candidates processed : {len(df_mentions):,}")
print(f"  Valid CNPJ (cnpj not null): {n_valid:,}")
print(f"  Rejected   (cnpj = null)  : {n_rejected:,}")

if n_rejected:
    print("\nRejections by reason:")
    display(
        df_mentions.filter(pl.col("cnpj").is_null())
        .group_by("rejected_reason")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
    )

df_valid = df_mentions.filter(pl.col("cnpj").is_not_null())
if n_valid:
    print("\nValid — by syntax pattern:")
    display(
        df_valid.group_by("syntax_pattern")
        .agg(
            pl.len().alias("n"),
            pl.col("pole_matches_syntax").mean().round(3).alias("pole_match_rate"),
        )
        .sort("n", descending=True)
    )
    print("\nValid — by confidence:")
    display(
        df_valid.group_by("confidence")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
    )

Total candidates processed : 124
  Valid CNPJ (cnpj not null): 104
  Rejected   (cnpj = null)  : 20

Rejections by reason:


rejected_reason,n
str,u32
"""cda_number""",15
"""natural_person""",5



Valid — by syntax pattern:


syntax_pattern,n,pole_match_rate
str,u32,f64
"""debtor_header_block""",49,1.0
"""debtor_requerido""",21,1.0
"""debtor_address_block""",18,1.0
"""debtor_executado_inline""",4,1.0
"""payment_creditor""",3,1.0
…,…,…
"""debtor_name_comma_cnpj""",1,1.0
"""substitution_creditor""",1,1.0
"""unclassified""",1,1.0



Valid — by confidence:


confidence,n
str,u32
"""high""",80
"""medium""",23
"""low""",1


## 3b. PF anonymization (derived view)

Raw `decisao` is **never modified**. We build a derived column `decisao_anonimizada` for
analysis and paper exports.

### Deterministic redaction (v1)

| Label | Source | Placeholder |
|-------|--------|-------------|
| CPF | regex | `[CPF]` |
| RG | regex | `[RG]` |
| OAB | regex | `[OAB]` |
| e-mail | regex | `[EMAIL]` |
| phone | regex | `[PHONE]` |
| PF party name | FACE `autores`/`reus` where `is_natural_person` | `[PESSOA]` |

### Model policy

- **CNPJ extraction** stays rule-based (`utils.cnpj_extraction`); models do not assign CNPJ ownership.
- **LegalBERT** (`dominguesm/legal-bert-ner-base-cased-ptbr`) is optional recall for `PESSOA` spans only.
- **Not used in production path:** `lgpd_pii_identifier` (WIP card), BERTopic (topic modeling, not span PII).
- PJ names are never redacted; PF names from FACE plus regex PII are redacted first.


In [5]:
redaction_rows: list[dict] = []
decision_redactions: list[dict] = []

for row in df_sample.iter_rows(named=True):
    result = anonymize_decision(
        row["decisao"],
        autores_raw=row.get("autores"),
        reus_raw=row.get("reus"),
    )
    decision_redactions.append(
        {
            "cd_processo": row["cd_processo"],
            "id_processo": str(row["id_processo"]),
            "n_spans": len(result.redaction_spans),
            "redaction_counts": str(result.redaction_counts),
            "decisao_preview": row["decisao"][:200].replace("\n", " "),
            "decisao_anonimizada_preview": result.decisao_anonimizada[:200],
        }
    )
    redaction_rows.extend(
        build_redaction_validation_rows(
            row["cd_processo"], str(row["id_processo"]), result
        )
    )

df_redaction_spans = pl.DataFrame(redaction_rows)
df_redaction_summary = pl.DataFrame(decision_redactions)
print(f"Redaction spans: {len(df_redaction_spans):,}")
if len(df_redaction_spans):
    display(
        df_redaction_spans.group_by("label", "source")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
    )
display(df_redaction_summary.head(5))


Redaction spans: 382


label,source,n
str,str,u32
"""pf_name""","""face_party""",365
"""cpf""","""regex""",12
"""phone""","""regex""",3
"""rg""","""regex""",2


cd_processo,id_processo,n_spans,redaction_counts,decisao_preview,decisao_anonimizada_preview
str,str,i64,str,str,str
"""2I000PCCM0000""","""1554848-49.2021.8.26.0090""",1,"""{'pf_name': 1}""","""SENTENÇA - MANDADO - OFÍCIO P…","""SENTENÇA - MANDADO - OFÍCIO P…"
"""2I000WK220000""","""1511642-77.2024.8.26.0090""",1,"""{'pf_name': 1}""","""SENTENÇA - MANDADO - OFÍCIO P…","""SENTENÇA - MANDADO - OFÍCIO P…"
"""2I000UVK90000""","""1516348-40.2023.8.26.0090""",1,"""{'pf_name': 1}""","""SENTENÇA - MANDADO - OFÍCIO P…","""SENTENÇA - MANDADO - OFÍCIO P…"
"""2I000P82U0000""","""1549314-27.2021.8.26.0090""",1,"""{'pf_name': 1}""","""SENTENÇA - MANDADO - OFÍCIO P…","""SENTENÇA - MANDADO - OFÍCIO P…"
"""2I000U8E90000""","""1590168-29.2022.8.26.0090""",1,"""{'pf_name': 1}""","""SENTENÇA - MANDADO - OFÍCIO P…","""SENTENÇA - MANDADO - OFÍCIO P…"


### Optional: LegalBERT recall layer

Requires `uv sync --group nlp`. Runs on a small sample only; compare extra `PESSOA` spans
against deterministic redaction.


In [6]:
if not legalbert_available():
    print("Install nlp group: uv sync --group nlp")
else:
    from utils.legalbert_ner import anonymize_with_legalbert

    sample = df_sample.head(3)
    for row in sample.iter_rows(named=True):
        base = anonymize_decision(
            row["decisao"],
            autores_raw=row.get("autores"),
            reus_raw=row.get("reus"),
        )
        with_model = anonymize_with_legalbert(
            row["decisao"],
            autores_raw=row.get("autores"),
            reus_raw=row.get("reus"),
        )
        print(row["cd_processo"], "base spans:", len(base.redaction_spans),
              "+legalbert:", len(with_model.redaction_spans))


/Users/etorebraga/Code/habitual-tax-debtor-research/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2135.37it/s]


2I000PCCM0000 base spans: 1 +legalbert: 2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1070.58it/s]


2I000WK220000 base spans: 1 +legalbert: 3


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1100.27it/s]


2I000UVK90000 base spans: 1 +legalbert: 2


In [7]:
redaction_path = OUTPUT_DIR / "pf_redaction_validation_sample.csv"
df_redaction_spans.write_csv(redaction_path)
print(f"Exported: {redaction_path} ({len(df_redaction_spans):,} spans)")


Exported: /Users/etorebraga/Code/habitual-tax-debtor-research/notebooks/playground/output/pf_redaction_validation_sample.csv (382 spans)


## 4. Manual review — stratified by syntax pattern

Fill `manual_label` in the exported CSV:

| Label | Meaning |
|-------|---------|
| `correct` | CNPJ belongs to `candidate_party` on `face_pole` |
| `incorrect` | CNPJ belongs to a different entity |
| `ambiguous` | Cannot determine from text |
| `third_party` | Bank or other non-party entity |

In [ ]:
review_cols = [
    "syntax_pattern",
    "expected_face_pole",
    "confidence",
    "rejected_reason",
    "cnpj",
    "party_from_syntax",
    "candidate_party",
    "face_pole",
    "pole_matches_syntax",
    "match_score",
    "snippet",
    "autores",
    "reus",
    "cd_processo",
]

df_valid = df_mentions.filter(pl.col("cnpj").is_not_null())
for pattern in df_valid["syntax_pattern"].unique().sort().to_list():
    subset = df_valid.filter(pl.col("syntax_pattern") == pattern).head(5)
    if subset.is_empty():
        continue
    print(f"\n=== {pattern} ({subset.height} shown) ===")
    display(subset.select(review_cols))

print("\n=== REJECTED (sample) ===")
display(df_mentions.filter(pl.col("cnpj").is_null()).head(5).select(review_cols))

In [ ]:
out_path = OUTPUT_DIR / "cnpj_validation_sample.csv"
export_cols = review_cols + [
    "id_processo",
    "assunto",
    "comarca",
    "method",
    "decision_preview",
    "manual_label",
    "notes",
]
df_mentions.select([c for c in export_cols if c in df_mentions.columns]).write_csv(out_path)
print(f"Exported: {out_path}  ({len(df_mentions):,} rows total)")

## 5. Precision after manual labeling

In [ ]:
labeled_path = OUTPUT_DIR / "cnpj_validation_sample.csv"
if labeled_path.exists():
    df_labeled = pl.read_csv(labeled_path)
    done = df_labeled.filter(pl.col("manual_label").str.len_chars() > 0)
    if done.is_empty():
        print("No labeled rows yet.")
    else:
        prec = (
            done.with_columns(
                pl.when(pl.col("manual_label").str.to_lowercase() == "correct")
                .then(1)
                .otherwise(0)
                .alias("is_correct")
            )
            .group_by("syntax_pattern", "confidence")
            .agg(
                pl.len().alias("n_labeled"),
                pl.col("is_correct").mean().round(3).alias("precision_correct"),
            )
            .sort("syntax_pattern", "confidence")
        )
        display(prec)
else:
    print("Run the export cell first.")

## 6. Lake coverage (CNPJ-like rate)

> **Note:** This cell scans all 6.1M decisions and typically takes 5–10 minutes.
> Run manually when you need a fresh coverage estimate; do not include in automated execution.

In [ ]:
stats = (
    lf_processos.select(
        pl.len().alias("n_decisions"),
        pl.col("decisao").str.contains(CNPJ_LIKE).sum().alias("n_with_cnpj_like"),
    )
    .collect()
    .with_columns(
        (pl.col("n_with_cnpj_like") / pl.col("n_decisions") * 100)
        .round(4)
        .alias("pct_cnpj_like")
    )
)
display(stats)